# Set up and run the framework
Use this notebook locally or on a remote GPU; edit the settings once, run the selected methods, and run the plotting cells when you want figures.

Prepare the project and its imports; on Google Colab this also downloads the latest committed code and installs notebook dependencies.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

ON_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
REPOSITORY_URL = "https://github.com/s4meer-git/BPDC_FL_Framework.git"

if ON_COLAB:
    PROJECT_ROOT = Path("/content/FLFrameWork")
    if (PROJECT_ROOT / ".git").exists():
        if "configuration" in sys.modules or "main" in sys.modules:
            raise RuntimeError("Restart the Python session before fetching code so old functions are not kept in memory.")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_ROOT / "JPNotebooks" / "requirements.txt")], check=True)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "JPNotebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "configuration.py").is_file():
    raise FileNotFoundError("Open this notebook from the project folder, or set PROJECT_ROOT to your checkout.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from configuration import configuration
from support_modules.experiment import prepare_runs, run_experiments, comparison_csvs

print("Project:", PROJECT_ROOT)
print("Device:", configuration().device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Results are saved as Results/<dataset>/RUN1, RUN2, RUN3/<method>.csv (one RUN per seed); on Google Colab the same files are copied to Drive after every round.

In [ ]:
RESULTS_ROOT = PROJECT_ROOT / "Results"
BACKUP_ROOT = None

if ON_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    BACKUP_ROOT = Path("/content/drive/MyDrive/FLFrameWork/Results")

print("Working results:", RESULTS_ROOT)
print("Backup:", BACKUP_ROOT if BACKUP_ROOT is not None else "Local run; CSVs stay on this computer.")

Choose datasets, methods and seeds here; the five-round smoke run uses one seed to check the setup before the full three-seed experiment.

In [ ]:
config = configuration()

DATASETS = ["cifar10"]
METHODS = ["fedavg", "fairrfl", "fedcda", "fedmut", "fcfl"]
SEEDS = [42, 43, 44]
SMOKE_SEEDS = [42]
SMOKE_TEST = True
RERUN_COMPLETED = False


# Common settings from the experiment setup; only the method-specific settings below change between methods.
config.num_of_clients = 30
config.rounds = 5 if SMOKE_TEST else 40
config.local_epochs = 5
config.batch_size = 64
config.lr = 0.01
config.dirichlet_alpha = 0.5
MOMENTUM = {"mnist": 0.5, "cifar10": 0.9, "tinyimagenet": 0.9}


# Keep this off for the setup's Table II; turn it on for the selfish-client comparison (0.05 = one selfish client).
config.selfishness = False
config.selfish_fraction = 0.05
config.selfish_strength = 0.7
config.selfish_start_round = 3


# Method-specific settings; to change them for a new experiment, first move or rename the existing dataset folder.
config.fairrfl_q = 1.0
config.fairrfl_tau = 2.5
config.fairrfl_dynamic_q = True
config.fairrfl_lipschitz = None
config.fedcda_warmup = 1 if SMOKE_TEST else 10
config.fcfl_alpha = 0.5
config.fcfl_fraction = 1.0

runs = prepare_runs(config, DATASETS, METHODS, RESULTS_ROOT, MOMENTUM, SMOKE_SEEDS if SMOKE_TEST else SEEDS, SMOKE_TEST)


# Every run writes one CSV here: Results/<dataset>/RUN<n>/<method>.csv, where RUN1 is the first seed in SEEDS.
print("Results folder:", RESULTS_ROOT)
for run_config, folder in runs:
    print(" ", folder.relative_to(RESULTS_ROOT) / f"{run_config.method}.csv", " (seed", str(run_config.seed) + ")")
if BACKUP_ROOT is not None:
    print("Drive copy of the same files:", BACKUP_ROOT)
print("Rounds:", config.rounds, "| Local epochs:", config.local_epochs, "| Seeds:", sorted({run[0].seed for run in runs}))
print("Selfish client IDs (counting from 0):", runs[0][0].selfish_client_ids)

Run the selected methods in order; completed runs are skipped, a method that diverges (NaN) is recorded and skipped, and incomplete runs restart from round 1 because CSVs are not model checkpoints.

In [ ]:
if ON_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime before training; loading CSVs and plotting do not need one.")

run_summary = run_experiments(runs, backup_root=BACKUP_ROOT, rerun_completed=RERUN_COMPLETED)
display(run_summary)

Optional diagnostic: inspect accuracy over rounds and FairRFL update sizes, including stopped runs, and save this check under temp rather than among final figures.

In [ ]:
from analysis.plots import plot_run_diagnostics

for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    diagnostic_path = PROJECT_ROOT / "temp" / "diagnostics" / folder.relative_to(RESULTS_ROOT) / "training_diagnostic.pdf"
    plot_run_diagnostics(csv_data, diagnostic_path)
    print("Diagnostic PDF:", diagnostic_path)

Run this cell when you want final client-accuracy plots and fairness summaries; only complete matching runs are accepted, and PDFs are saved beside their CSVs.

In [ ]:
from analysis.plots import final_client_scores, plot_client_test_accuracy
from support_modules.experiment import backup_csv, backup_path

for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    scores, settings = final_client_scores(csv_data)
    pdf_path = folder / "final_test_accuracy_all_methods.pdf"
    summary = plot_client_test_accuracy(scores, settings, pdf_path)
    display(summary)
    backup_folder = backup_path(folder, BACKUP_ROOT)
    backup_csv(pdf_path, backup_folder)
    print("Saved PDF:", pdf_path)
    if backup_folder is not None:
        print("Drive folder:", backup_folder)

Run this after every seed has finished to print Table II from the setup: final-round metrics as mean ± sample standard deviation over the seeds, saved as summary.csv beside the seed folders.

In [ ]:
from analysis.results import seed_summary
from analysis.plots import format_seed_summary
from support_modules.experiment import backup_csv, backup_path

comparisons = {}
for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    comparisons.setdefault(folder.parent, {})[folder.name] = csv_data

for comparison_folder, csv_by_seed in comparisons.items():
    means, deviations, per_seed, partial = seed_summary(csv_by_seed)
    summary = format_seed_summary(means, deviations)
    summary_path = comparison_folder / "summary.csv"
    summary.to_csv(summary_path)
    per_seed.to_csv(comparison_folder / "summary_per_seed.csv", index=False)
    print(comparison_folder.relative_to(RESULTS_ROOT), "| seeds:", sorted(int(seed) for seed in per_seed["seed"].unique()))
    for method, missing in partial.items():
        print(f"{method} diverged for seeds {missing}; it is left out of the mean ± std table.")
    display(summary)
    backup_folder = backup_path(comparison_folder, BACKUP_ROOT)
    backup_csv(summary_path, backup_folder)
    backup_csv(comparison_folder / "summary_per_seed.csv", backup_folder)
    print("Saved Table II:", summary_path)

Optional: choose one method and generate its final client-accuracy plot in each selected dataset folder.

In [ ]:
from support_modules.experiment import backup_csv, backup_path

METHOD = "fairrfl"

for folder, csv_data in comparison_csvs(runs, backup_root=BACKUP_ROOT):
    scores, settings = final_client_scores(csv_data)
    if METHOD not in scores.columns:
        raise ValueError(f"Choose a loaded method: {list(scores.columns)}")
    pdf_path = folder / f"final_test_accuracy_{METHOD}.pdf"
    display(plot_client_test_accuracy(scores[[METHOD]], settings, pdf_path))
    backup_folder = backup_path(folder, BACKUP_ROOT)
    backup_csv(pdf_path, backup_folder)
    print("Saved PDF:", pdf_path)

Selfish runs use <dataset>_selfish and smoke tests smoke_<dataset>, so they never mix with the full baseline; to rerun with different settings, first move or rename the existing dataset folder; one of 30 clients is selected at fraction 0.05 (0.3 gives nine). Keep the same seed and shared settings for both scenarios. No setting guarantees a particular ranking; report the measured results. Compare honest-client mean, worst accuracy and standard deviation together. After pulling new Python code, restart the kernel and rerun setup before training.